In [17]:
import pandas as pd
import numpy as np
import faiss
import json
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent
PRODUCTS_PATH = PROJECT_ROOT / "data" / "processed" / "products.parquet"
print(f"Loading products from: {PRODUCTS_PATH}")

df = pd.read_parquet(PRODUCTS_PATH)
print(f"Total products in raw dataframe: {len(df):,}")


Loading products from: d:\goal\programming\quera\boot_camp\project3\data\processed\products.parquet
Total products in raw dataframe: 1,283,496


In [18]:
# # Cell 3 — Load the BAAI/bge-m3 dense embedding model

# import os
# from pathlib import Path

# os.environ["USE_TF"] = "0"
# os.environ["USE_FLAX"] = "0"

# import torch
# from sentence_transformers import SentenceTransformer

# MODEL_NAME = "BAAI/bge-m3"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # The notebook is executed from project3/notebook.
# # Store the reusable model cache at project3/artifacts/model_cache.
# PROJECT_ROOT = Path.cwd().resolve().parent
# MODEL_CACHE_DIR = PROJECT_ROOT / "artifacts" / "model_cache"
# MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# print(f"Loading model: {MODEL_NAME}")
# print(f"Device: {DEVICE}")
# print(f"Cache directory: {MODEL_CACHE_DIR}")

# dense_model = SentenceTransformer(
#     MODEL_NAME,
#     cache_folder=str(MODEL_CACHE_DIR),
#     device=DEVICE,
#     token=False,  # Ignore any stale/invalid Hugging Face token saved locally
# )

# print("\nModel loaded successfully.")
# print("Embedding dimension:", dense_model.get_sentence_embedding_dimension())
# print("Maximum sequence length:", dense_model.max_seq_length)


In [20]:
# Use the project-local Hugging Face cache.
root_dir = Path("..").resolve()
model_cache_dir = root_dir / "artifacts" / "model_cache"
model_cache_repo_dir = model_cache_dir / "models--BAAI--bge-m3"
snapshots_dir = model_cache_repo_dir / "snapshots"

print(f"Cache directory: {model_cache_dir}")
print(f"Snapshots directory: {snapshots_dir}")

if not snapshots_dir.exists():
    raise FileNotFoundError(
        "The local BGE-M3 cache was not found. "
        f"Expected snapshots at: {snapshots_dir}"
    )

# A Hugging Face cache may contain one or more revision snapshots.
# Select the newest snapshot that contains the core model config.
snapshot_candidates = sorted(
    [
        path for path in snapshots_dir.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not snapshot_candidates:
    raise FileNotFoundError(
        "No complete local BGE-M3 snapshot was found. "
        "Expected at least a snapshot containing config.json."
    )

local_model_path = snapshot_candidates[0]

print(f"Using local model snapshot: {local_model_path}")
print("Network access is disabled for this load.")

device = "cuda" if torch.cuda.is_available() else "cpu"

dense_model = SentenceTransformer(
    str(local_model_path),
    device=device,
    local_files_only=True,
    token=False,
)

print("\nModel loaded successfully from local cache.")
print(f"Device: {device}")
print(f"Embedding dimension: {dense_model.get_sentence_embedding_dimension()}")
print(f"Maximum sequence length: {dense_model.max_seq_length}")


Cache directory: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache
Snapshots directory: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache\models--BAAI--bge-m3\snapshots
Using local model snapshot: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181
Network access is disabled for this load.

Model loaded successfully from local cache.
Device: cuda
Embedding dimension: 1024
Maximum sequence length: 8192


In [21]:
# 1. Load data if not already present in memory
if 'df_products' not in globals():
    products_path = root_dir / "data" / "processed" / "products.parquet"
    print(f"Loading products from: {products_path}")
    df_products = pd.read_parquet(products_path)

print(f"Total products in raw dataframe: {len(df_products):,}")

# 2. Select a stable subset for efficient local dense indexing (e.g. 1000 items for robust validation)
# Using a fixed random_state for full reproducibility
SAMPLE_SIZE = 1000
df_sample = df_products.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Working dense corpus sample size: {len(df_sample)}")

# 3. Construct rich text representation for each product
# Combining title, brand, and categories to give dense model maximum semantic context
def make_product_text(row):
    parts = []
    if pd.notna(row.get('title_fa')):
        parts.append(str(row['title_fa']))
    if pd.notna(row.get('Brand')):
        parts.append(f"برند: {row['Brand']}")
    if pd.notna(row.get('Category1')):
        parts.append(f"دسته‌بندی: {row['Category1']}")
    if pd.notna(row.get('Category2')):
        parts.append(f" زیردسته: {row['Category2']}")
    return " - ".join(parts)

corpus_texts = df_sample.apply(make_product_text, axis=1).tolist()
product_ids = df_sample['id'].tolist()

print("\nSample constructed text #0:")
print(corpus_texts[0])
print(f"Associated Product ID: {product_ids[0]}")


Total products in raw dataframe: 1,283,496
Working dense corpus sample size: 1000

Sample constructed text #0:
صندل مردانه پویان گام پیام مدل داریوش کد 10645 - برند: پویان گام پیام - دسته‌بندی: کفش مردانه -  زیردسته: صندل مردانه
Associated Product ID: 11728451


In [22]:
print("Generating embeddings for the corpus sample (1000 items)...")
# encode returns a numpy array of shape (N, 1024)
corpus_embeddings = dense_model.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True # Recommended for cosine similarity via inner product in FAISS
)

embedding_dim = corpus_embeddings.shape[1]
print(f"Generated embeddings shape: {corpus_embeddings.shape}")

# 2. Build FAISS Index (IndexFlatIP for Inner Product since embeddings are normalized -> Cosine Similarity)
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(corpus_embeddings.astype(np.float32))

print(f"FAISS index total vectors: {faiss_index.ntotal}")
print("FAISS Index successfully built and ready for dense search.")


Generating embeddings for the corpus sample (1000 items)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches: 100%|██████████| 16/16 [06:22<00:00, 23.90s/it]

Generated embeddings shape: (1000, 1024)
FAISS index total vectors: 1000
FAISS Index successfully built and ready for dense search.


In [23]:
def dense_search(query: str, top_k: int = 5):
    """
    Search top_k most semantically similar products for a given query text.
    """
    # 1. Encode and normalize the single query vector
    query_emb = dense_model.encode(
        [query],
        normalize_embeddings=True
    ).astype(np.float32)
    
    # 2. Query FAISS index for top_k results
    scores, indices = faiss_index.search(query_emb, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        row = df_sample.iloc[idx]
        results.append({
            "product_id": int(row['id']),
            "title_fa": row['title_fa'],
            "brand": row.get('Brand', None),
            "category": row.get('Category1', None),
            "score": float(score)
        })
    return results

# تست کیفی با یک کوئری نمونه (مثلاً کفش یا صندل بر اساس داده‌ای که در نمونه دیدیم)
test_query = "کفش و صندل مردانه راحت"
print(f"--- Query: '{test_query}' ---")
hits = dense_search(test_query, top_k=3)

for rank, item in enumerate(hits, start=1):
    print(f"Rank {rank} (Score: {item['score']:.4f}): [ID: {item['product_id']}] {item['title_fa']} | برند: {item['brand']}")


--- Query: 'کفش و صندل مردانه راحت' ---
Rank 1 (Score: 0.6636): [ID: 5333787] کفش راحتی مردانه مدل 0830804 | برند: متفرقه
Rank 2 (Score: 0.6241): [ID: 7092441] کفش راحتی مردانه مدل JB1255 | برند: متفرقه
Rank 3 (Score: 0.6195): [ID: 9713132] کفش راحتی مردانه مدل WH.1486 | برند: متفرقه


In [24]:
# 1. Select 100 benchmark items from the indexed sample for quantitative evaluation (Method A)
BENCHMARK_SIZE = 100
benchmark_df = df_sample.sample(n=min(BENCHMARK_SIZE, len(df_sample)), random_state=42).reset_index(drop=True)

print(f"Number of benchmark queries for evaluation: {len(benchmark_df)}")

# 2. Evaluation loop: Compute Hit@1, Hit@5, and MRR
hit_at_1_count = 0
hit_at_5_count = 0
reciprocal_ranks = []

for idx, row in benchmark_df.iterrows():
    target_id = int(row['id'])
    # In Method A, the query is derived from the real product title
    query_text = row['title_fa']
    
    # Retrieve top 5 candidates
    results = dense_search(query_text, top_k=5)
    retrieved_ids = [res['product_id'] for res in results]
    
    # Check Hit@1
    if len(retrieved_ids) > 0 and retrieved_ids[0] == target_id:
        hit_at_1_count += 1
        
    # Check Hit@5 and Reciprocal Rank
    if target_id in retrieved_ids:
        hit_at_5_count += 1
        rank = retrieved_ids.index(target_id) + 1  # 1-based rank
        reciprocal_ranks.append(1.0 / rank)
    else:
        reciprocal_ranks.append(0.0)

# 3. Calculate final aggregate metrics
total_queries = len(benchmark_df)
hit_rate_at_1 = (hit_at_1_count / total_queries) * 100
hit_rate_at_5 = (hit_at_5_count / total_queries) * 100
mrr = sum(reciprocal_ranks) / total_queries

print("\n" + "="*45)
print("--- Dense Baseline Quantitative Evaluation ---")
print("="*45)
print(f"Total Benchmark Queries : {total_queries}")
print(f"Hit Rate @ 1            : {hit_rate_at_1:.2f}%")
print(f"Hit Rate @ 5            : {hit_rate_at_5:.2f}%")
print(f"MRR (Mean Reciprocal Rank): {mrr:.4f}")
print("="*45)


Number of benchmark queries for evaluation: 100

--- Dense Baseline Quantitative Evaluation ---
Total Benchmark Queries : 100
Hit Rate @ 1            : 100.00%
Hit Rate @ 5            : 100.00%
MRR (Mean Reciprocal Rank): 1.0000


In [25]:
# Realistic Method A: Simulating real user search queries from product titles
# (Users typically search short prefixes or main product keywords, not full descriptions)

realistic_hit_at_1 = 0
realistic_hit_at_5 = 0
realistic_mrr = []
failed_examples = []

for idx, row in benchmark_df.iterrows():
    target_id = int(row['id'])
    full_title = str(row['title_fa'])
    
    # Simulate user query: take first 3 to 5 words of the title
    words = full_title.split()
    query_text = " ".join(words[:min(4, len(words))]) if len(words) >= 2 else full_title
    
    results = dense_search(query_text, top_k=5)
    retrieved_ids = [res['product_id'] for res in results]
    
    # Evaluate
    if len(retrieved_ids) > 0 and retrieved_ids[0] == target_id:
        realistic_hit_at_1 += 1
        
    if target_id in retrieved_ids:
        realistic_hit_at_5 += 1
        rank = retrieved_ids.index(target_id) + 1
        realistic_mrr.append(1.0 / rank)
    else:
        realistic_mrr.append(0.0)
        failed_examples.append({
            "target_id": target_id,
            "simulated_query": query_text,
            "full_title": full_title,
            "top_retrieved": results[0]['title_fa'] if results else "None"
        })

real_hit_1_pct = (realistic_hit_at_1 / total_queries) * 100
real_hit_5_pct = (realistic_hit_at_5 / total_queries) * 100
real_mrr_val = sum(realistic_mrr) / total_queries

print("="*55)
print("--- Realistic Dense Baseline Evaluation (Short Queries) ---")
print("="*55)
print(f"Total Benchmark Queries : {total_queries}")
print(f"Hit Rate @ 1            : {real_hit_1_pct:.2f}%")
print(f"Hit Rate @ 5            : {real_hit_5_pct:.2f}%")
print(f"MRR                     : {real_mrr_val:.4f}")
print("="*55)

if failed_examples:
    print(f"\nExample of a Missed Target (Total Missed: {len(failed_examples)}):")
    sample_fail = failed_examples[0]
    print(f"Simulated Query : '{sample_fail['simulated_query']}'")
    print(f"Target Title    : '{sample_fail['full_title']}'")
    print(f"Top Retrieved   : '{sample_fail['top_retrieved']}'")


--- Realistic Dense Baseline Evaluation (Short Queries) ---
Total Benchmark Queries : 100
Hit Rate @ 1            : 78.00%
Hit Rate @ 5            : 91.00%
MRR                     : 0.8282

Example of a Missed Target (Total Missed: 9):
Simulated Query : 'تی شرت آستین کوتاه'
Target Title    : 'تی شرت آستین کوتاه زنانه طرح دریم کچر مدل L002'
Top Retrieved   : 'تی شرت آستین کوتاه مردانه مدل  114 رنگ طوسی'


In [26]:
# Compile evaluation summary dictionary
evaluation_summary = {
    "model_name": "BAAI/bge-m3",
    "embedding_dim": embedding_dim,
    "similarity_metric": "Cosine Similarity (FAISS IndexFlatIP)",
    "corpus_size": len(df_sample),
    "benchmark_size": total_queries,
    "metrics": {
        "exact_title": {
            "hit_rate_at_1": hit_rate_at_1,
            "hit_rate_at_5": hit_rate_at_5,
            "mrr": mrr
        },
        "realistic_short_query": {
            "hit_rate_at_1": real_hit_1_pct,
            "hit_rate_at_5": real_hit_5_pct,
            "mrr": real_mrr_val,
            "missed_count": len(failed_examples)
        }
    }
}

# Save results for Day 3 and Day 4 comparison
eval_output_dir = root_dir / "evaluation"
eval_output_dir.mkdir(parents=True, exist_ok=True)
eval_output_file = eval_output_dir / "dense_baseline_results.json"

with open(eval_output_file, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, ensure_ascii=False, indent=2)

print(f"Results successfully saved to: {eval_output_file}")
print("\n" + "="*50)
print("DAY 2 - MILESTONE 3 (Dense Baseline) COMPLETED!")
print("="*50)
print(f"Dense Hit@5 (Realistic) : {real_hit_5_pct:.2f}%")
print(f"Dense MRR   (Realistic) : {real_mrr_val:.4f}")
print("Notebook is clean, self-contained, and ready for team review.")


Results successfully saved to: D:\goal\programming\quera\boot_camp\project3\evaluation\dense_baseline_results.json

DAY 2 - MILESTONE 3 (Dense Baseline) COMPLETED!
Dense Hit@5 (Realistic) : 91.00%
Dense MRR   (Realistic) : 0.8282
Notebook is clean, self-contained, and ready for team review.


In [27]:
print("PyTorch CUDA available:", torch.cuda.is_available())
print("Model device:", dense_model.device)
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("Model parameter device:", next(dense_model.parameters()).device)


PyTorch CUDA available: True
Model device: cuda:0
CUDA device: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Model parameter device: cuda:0
